# Exercise 01: Classical Text Retrieval

> **Chapter:** Ch01 - Classical Text Retrieval
> **Estimated time:** ~55 minutes
>
> This exercise is optional. No submission, no grading.
> The solution notebook is released one week after the exercise.

Classical text retrieval turns raw text into terms, then ranks documents by how well their terms
match a query. This exercise takes you along that whole path. First you reason about a BM25
ranking that looks wrong. Then you build the two halves of a real search engine over a collection
of movies: the text-extraction pipeline that decides what counts as a term, and the BM25 scorer
that ranks the results. You will feel how much the pipeline alone decides whether a query finds
anything at all.

**Your tasks** (tick them off as you go):

- [ ] **Quiz** - 20 questions in the quiz app
- [ ] ✏ **T1: The short document wins** - explain a BM25 ranking that looks wrong
- [ ] 💻 **C1: Make the queries find their movies** - write the text-extraction pipeline (in this notebook)
- [ ] 💻 **C2: A BM25 search engine** - implement the scorer (in `tasks.py`)
- [ ] ✏ **T2: Scaling to a million movies** - reason about performance at scale

---

## Quiz

Open the quiz app and work through the **20 questions** for this chapter. On the start screen,
pick the topic **"01 - Classical Text Retrieval"**:

**Quiz app:** https://roger-weber.github.io/mmir-unibasel-hs26/quiz/

The quiz covers definitions and basic concepts. The tasks below go further: they ask you to
explain a ranking that looks wrong, and to build and verify a search engine yourself.

---

## Understanding BM25 rankings

> **How this works:** Write your answer in the markdown cell below each question (replace
> *Your answer here...*). The solution notebook fills the same cell with a model answer, so you
> can compare and reflect.

### ✏ Task T1 - The short document wins

Here is the book's BM25 ranking for the query `cat dog forest` on the 12-document MINI collection,
computed with **k1 = 1.2**, **b = 0.75**, and average document length **avgdl = 9.67**. The third
column counts the occurrences of (`cat`, `dog`, `forest`) in each document:

| Rank | Doc | Length | tf (cat, dog, forest) | BM25 |
|---|---|---:|:---:|---:|
| 1 | b9 | 3 | (1, 1, 1) | 2.930 |
| 2 | b1 | 8 | (2, 2, 1) | 2.836 |
| 3 | b10 | 14 | (2, 2, 2) | 2.568 |

Document **b9** wins even though it contains the *fewest* query-term occurrences: b1 and b10 both
contain strictly more of every query term, yet rank below it.

1. Explain **which part of the BM25 formula** produces this outcome, and trace it using the
   numbers above.
2. **Predict** how the ordering of b9, b1, and b10 changes if you set **b = 0**, and justify your
   prediction. You will be able to check it against your own scorer later in the notebook.
3. You control one document and want it to rank first for `cat dog forest`, with k1 and b fixed at
   1.2 and 0.75. **How would you craft that document** to attack the scoring function, and where
   are the limits of what such an attack can gain?

> *Your answer here...*

---

## The movie collection

From here on you work with a real collection: about 500 movies, each with a title, plot overview,
tagline, top cast, and genres combined into one searchable text. This is the same `movies` dataset
used in the demos; the first load downloads it (a few seconds).

Run this cell first. It makes the `shared/` package importable regardless of where the notebook
lives, then loads the collection.

In [ ]:
# Standard setup - run this first
import sys, pathlib
from collections import Counter

# Make `shared/` importable regardless of notebook depth
# (exercise notebooks live in exercises/chNN/, solutions in exercises/chNN/solution/).
_p = pathlib.Path().resolve()
while not (_p / "shared").is_dir() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

%load_ext autoreload
%autoreload 2

from shared.collections import load_collection
from shared.display import print_table, display_md

movies = load_collection("movies-small")
movie_titles = {doc["id"]: doc["title"] for doc in movies}   # id -> title
movies_text = {doc["id"]: doc["text"] for doc in movies}     # id -> full searchable text

print(f"Loaded {len(movies)} movies.")
print("Example title:", movie_titles[next(iter(movie_titles))])

## From text to search terms

> **How this works:** Implement the function in the stub cell (replace `# YOUR CODE HERE`). Run the
> verify cell; the assertions either pass or fail with a specific message. Write the code yourself
> or direct an AI to implement it; the goal is to understand *why* a given pipeline succeeds.

### 💻 Task C1 - Make the queries find their movies

A search engine only ever matches the *terms* it extracts, so the extraction pipeline quietly
decides what can be found. Your job is to turn a raw string into a list of terms so that a handful
of title searches return the movies a user obviously means.

The file **`tasks.py`** in this folder provides several small text-processing functions. Open it
and read what each one does, then compose them into `extract_terms` below so that **every assertion
in the verify cell passes**. Run the verify cell and adjust your pipeline until it does.

In [ ]:
import tasks   # the text-processing functions live here - open the file and read them

def extract_terms(text: str) -> list[str]:
    """Turn a raw string into a list of search terms by composing functions from `tasks`."""
    # YOUR CODE HERE
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# --- Given: a tiny title-only search built on your extract_terms ---
def find_movies(query: str) -> set[str]:
    """Titles whose extracted terms contain every extracted query term."""
    q = {t for t in extract_terms(query) if t}
    if not q:
        return set()
    return {title for title in movie_titles.values() if q <= set(extract_terms(title))}

In [ ]:
# --- Verify C1 ---
r1 = find_movies("star wars")
assert "Star Wars" in r1, f"query 'star wars' should retrieve 'Star Wars'; got {sorted(r1)[:5]}"

r2 = find_movies("toys")
assert "Toy Story" in r2, f"query 'toys' should retrieve 'Toy Story'; got {sorted(r2)[:5]}"

r3 = find_movies("it")
assert "It Takes Two" in r3, f"query 'it' should retrieve 'It Takes Two'; got {sorted(r3)[:5]}"

print("✓ all queries return the movie the user meant!")

Keep the `extract_terms` you arrived at: the search engine below reuses it to score full documents.

---

## Building a BM25 search engine

> **How this works:** Implement the `BM25Scorer` class in `tasks.py`. You can write the code
> yourself or direct an AI to implement it from the spec. After saving `tasks.py`, just re-run the
> verify cell - autoreload picks up your changes automatically, no kernel restart needed.

### 💻 Task C2 - A BM25 search engine over the movies

You now have a pipeline that decides what counts as a term. Wrap it in a ranking engine. Implement
`BM25Scorer` in `tasks.py` so it indexes the movies once and answers ranked queries end to end,
from raw query string to ranked movie ids. Use the **positive (Lucene) IDF** and the BM25 formula
from Task T1.

Interface:

- `BM25Scorer(documents: dict[str, str], extract_terms, k1: float = 1.2, b: float = 0.75)` -
  `documents` maps a document id to its raw text; `extract_terms` is the term-extraction function
  you built above. The constructor extracts terms for every document and precomputes the
  document frequencies, the document count `N`, and the average document length `avgdl`.
- `idf(term: str) -> float` - positive BM25 IDF, $\log\!\big(1 + \frac{N - \text{df} + 0.5}{\text{df} + 0.5}\big)$.
- `search(query: str, k: int = 10) -> list[tuple[str, float]]` - extract the query's terms, score
  every document, keep only positive scores, and return the top `k` as `(doc_id, score)` pairs
  sorted by descending score, breaking ties by ascending id. An empty query, or one whose terms
  match nothing, returns `[]`.

In [ ]:
# --- Setup C2 ---
from tasks import BM25Scorer

scorer = BM25Scorer(movies_text, extract_terms)   # reuses the pipeline you built above

def show(query, k=5):
    display_md(f"**Top {k} for `{query}`:**")
    print_table(
        [[f"{score:.3f}", movie_titles[doc_id]] for doc_id, score in scorer.search(query, k=k)],
        headers=["BM25", "Title"],
    )

show("toy story")

In [ ]:
# --- Verify C2 ---
def top_title(query):
    res = scorer.search(query, k=1)
    return movie_titles[res[0][0]] if res else None

# The obvious top hit for a clear query.
assert top_title("toy story") == "Toy Story", f"got {top_title('toy story')}"
assert top_title("jurassic park dinosaurs") == "Jurassic Park", f"got {top_title('jurassic park dinosaurs')}"

# Ranking contract.
res = scorer.search("love", k=5)
assert len(res) == 5, f"k should cap results at 5, got {len(res)}"
scores = [s for _, s in res]
assert scores == sorted(scores, reverse=True), "results must be sorted by descending score"

# Edge cases.
assert scorer.search("", k=5) == [], "empty query should return []"
assert scorer.search("qzzxwv", k=5) == [], "a query matching nothing should return []"

print("✓ BM25Scorer passes all checks!")

In [ ]:
# --- Explore C2 ---
# Try your own queries. Which query produces the most surprising top result, and can you
# explain it from the pipeline (what got stemmed or folded) and the BM25 factors (rare terms,
# document length)?
show("space adventure with aliens")
show("romance in paris")

### ✏ Task T2 - Scaling to a million movies

Your `search` method scores **every** document in the collection for **every** query. That is fine
for 500 movies, but imagine the collection grows to 1,000,000 movies and the engine must answer
many queries per second.

Explain why the current approach becomes too slow, and describe a better strategy that avoids
touching most documents for a typical short query. You have not studied the relevant data structure
yet (it comes in a later chapter), so reason from first principles: given a query term, what would
you want to look up instantly, and what would you precompute to make that possible?

> *Your answer here...*